In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import polars as pl
from datetime import datetime, timedelta
from matplotlib.dates import MonthLocator
import matplotlib.dates as mdates 
import matplotlib.transforms as mtransforms
import scienceplots
import seaborn as sns
cmp = sns.color_palette('Set2')
import oracledb
from matplotlib.dates import MonthLocator
from matplotlib.dates import DateFormatter

In [ ]:
run = 1

if run:
    def query_full(): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        INSERT QUERY FOR ALL POSITIVE TESTS BETWEEN 01-09-2020 AND 31-12-2021
        '''
        return([q1])

    def retrieve_data(query):
        # extract data from DB for all pids
        oracledb.init_oracle_client()
        with oracledb.connect(user='', dsn='') as con:
            with con.cursor() as cur:
                out = [np.asarray(cur.execute(qq).fetchall()) for qq in query()]
                cols = cur.description
        return(out, cols)



    data_full, columns_full = retrieve_data(query_full)
    cols_full = [col[0] for col in columns_full]

    df_full = pd.DataFrame(data_full[0], columns=cols_full)
    df_full = pl.from_pandas(df_full)

In [ ]:
def query_school(): 
# specify your OracleSQL query (or multiple ones) 
    q1 = f'''
        QUERY NATION-SCALE NETWORK (SCHOOL LAYER) HERE
        '''
    return([q1])

def query_work():
    q1 = f'''
    QUERY NATION-SCALE NETWORK (WORKPLACE LAYER) HERE
    '''
    return([q1])

print('Reading Schools Dataset')
data_school, columns_school = retrieve_data(query_school)
cols_school = [col[0] for col in columns_school]

df_school = pd.DataFrame(data_school[0][:,:], columns=cols_school[:])
df_school = pl.from_pandas(df_school)

print('Reading Workplace Dataset')
data_work, columns_work = retrieve_data(query_work)
cols_work = [col[0] for col in columns_work]

df_work = pd.DataFrame(data_work[0][:, :], columns=cols_work[:])
df_work = pl.from_pandas(df_work)

In [ ]:
pcrs_df = df_full.filter(pl.col('CASEDEF') == 'SARS2')
pos_pcrs_by_day = pcrs_df.group_by('PRDATE_ADJUSTED').len().sort('PRDATE_ADJUSTED')
pos_pcrs_by_day = pos_pcrs_by_day[3:-5]


In [ ]:
pcrs_school_ids = pcrs_df.join(df_school, on = 'PERSON_ID', how = 'left').with_columns(school_during_sample = 
                                         ((pl.col('ELEV3_VFRA') <= pl.col('PRDATE_ADJUSTED'))& 
                                         (pl.col('ELEV3_VTIL') >= pl.col('PRDATE_ADJUSTED'))))
pcrs_school = pcrs_school_ids.filter((pl.col('school_during_sample')))

pcrs_not_school = pcrs_school_ids.filter((~pl.col('school_during_sample') )| (pl.col('school_during_sample').is_null())) 
                                         
pcrs_not_school_unique = pcrs_not_school.select(pl.col('ROWID')).unique()
pcrs_school_unique = pcrs_school.select(pl.col('ROWID')).unique()
intersection_ids = set(pcrs_not_school_unique.to_numpy().flatten()).intersection(set(pcrs_school_unique.to_numpy().flatten()))
pcrs_not_school = pcrs_not_school.filter(~pl.col('ROWID').is_in(intersection_ids))
people_tests_school = pcrs_school.select(pl.col('PERSON_ID', 'PRDATE_ADJUSTED')).unique() 
people_tests_not_school = pcrs_not_school.select(pl.col('PERSON_ID', 'PRDATE_ADJUSTED')).unique()

In [ ]:
cases_school = people_tests_school.group_by('PRDATE_ADJUSTED').len().filter((pl.col('PRDATE_ADJUSTED') >= 
                                                                             datetime.strptime('2020-09-01', '%Y-%m-%d')) & 
                                                                            (pl.col('PRDATE_ADJUSTED') < 
                                                                             datetime.strptime('2022-01-01', '%Y-%m-%d'))).rename({'len' : 'count'})
cases_not_school = people_tests_not_school.group_by('PRDATE_ADJUSTED').len().filter((pl.col('PRDATE_ADJUSTED') >= 
                                                                             datetime.strptime('2020-09-01', '%Y-%m-%d')) & 
                                                                            (pl.col('PRDATE_ADJUSTED') < 
                                                                             datetime.strptime('2022-01-01', '%Y-%m-%d'))).rename({'len' : 'count'})
cases_school = cases_school.sort('PRDATE_ADJUSTED')
cases_not_school = cases_not_school.sort('PRDATE_ADJUSTED')

In [ ]:
def query_lifelines():
    q1 = f'''
    QUERY LIFELINES DATABASE FOR BIRTH DATES HERE
    '''
    
    return([q1])

data_lifelines, columns_lifelines = retrieve_data(query_lifelines)
cols_lifelines = [col[0] for col in columns_lifelines]

df_lifelines = pd.DataFrame(data_lifelines[0][:,:], columns=cols_lifelines[:])
df_lifelines = pl.from_pandas(df_lifelines)

In [ ]:
metadata_all = pl.read_csv('../../../cleaned_2020_to_2024_sequences/final_merged_new_id_QC_unique_lineage_metadata.csv')
metadata_all_ages = metadata_all.join(df_lifelines, on = 'PERSON_ID', how = 'left')
metadata_all_ages = metadata_all_ages.with_columns(Age_at_testing = 
                                                   ((pl.col('DATESAMPLING').str.strptime(pl.Date,
                                                                                         strict = False).cast(pl.Date)-
                                           pl.col('BIRTHDAY').cast(pl.Date)).dt.total_days()/365).cast(pl.Int64))
metadata_all_ages = metadata_all_ages.filter((pl.col('DATESAMPLING') >= '2020-09-01') & 
                                             (pl.col('DATESAMPLING') <= '2021-12-31'))
metadata_all_ages = metadata_all_ages.with_columns(Age_group = 
    pl.when(pl.col('Age_at_testing') >=60).then(
        pl.lit('60+')
    ).otherwise(pl.when(pl.col('Age_at_testing').is_between(40, 59)).then(
    pl.lit('40-59')
    ).otherwise(pl.when(pl.col('Age_at_testing').is_between(19, 39)).then(
    pl.lit('19-39')
    ).otherwise(pl.when(pl.col('Age_at_testing').is_between(11, 18)).then(
        pl.lit('11-18'))
                 .otherwise(pl.when(pl.col('Age_at_testing').is_between(0, 10)).then(
                 pl.lit('0-10'))
               )))))

In [ ]:
metadata = pd.read_csv('../../newids_combined_metadata_ID_seqname_filename_pango.csv')
metadata = pl.from_pandas(metadata).with_columns(pl.col('PERSON_ID').cast(pl.Int64))
metadata = metadata.filter((pl.col('SampleDateTime')>='2020-09-01') & (pl.col('SampleDateTime')<'2022-01-01'))

In [ ]:
school_metadata = metadata_all_ages.join(df_school, on ='PERSON_ID', how = 'left').with_columns(pl.col('DATESAMPLING').
                                                                                            str.strptime(pl.Date))
school_metadata = school_metadata.with_columns(school_during_sample = 
                                         ((pl.col('ELEV3_VFRA') <= pl.col('DATESAMPLING'))& 
                                         (pl.col('ELEV3_VTIL') >= pl.col('DATESAMPLING'))))
sequences_school = school_metadata.filter((pl.col('school_during_sample')))
sequences_not_school = school_metadata.filter((~pl.col('school_during_sample') )| 
                                              (pl.col('school_during_sample').is_null())) 
beep = sequences_not_school.select(pl.col('strain')).unique()
boop = sequences_school.select(pl.col('strain')).unique()
intersection_ids = set(beep.to_numpy().flatten()).intersection(set(boop.to_numpy().flatten()))
sequences_not_school = sequences_not_school.filter(~pl.col('strain').is_in(intersection_ids))

In [ ]:
people_sequences_not_school = sequences_not_school.select(pl.col('strain', 'DATESAMPLING')).unique()
people_sequences_school = sequences_school.select(pl.col('strain', 'DATESAMPLING')).unique()
sequence_counts_school = people_sequences_school.group_by('DATESAMPLING').len().sort('DATESAMPLING').rename({'len' : 'count'})
sequence_counts_not_school = people_sequences_not_school.group_by('DATESAMPLING').len().sort('DATESAMPLING').rename({'len' : 'count'})

counts_not_school = cases_not_school.with_columns(pl.col('PRDATE_ADJUSTED').cast(pl.Date)).join(sequence_counts_not_school,
                                                                            left_on = 'PRDATE_ADJUSTED', 
                                                                           right_on = 'DATESAMPLING', 
                                                                            how = 'left', 
                                                                           suffix = '_sequences').fill_null(0)

counts_school = cases_school.with_columns(pl.col('PRDATE_ADJUSTED').cast(pl.Date)).join(sequence_counts_school,
                                                                            left_on = 'PRDATE_ADJUSTED', 
                                                                           right_on = 'DATESAMPLING', 
                                                                            how = 'left', 
                                                                           suffix = '_sequences').fill_null(0)

In [ ]:
prop_not_school = (counts_not_school.select('count_sequences').to_numpy().flatten() / 
                   counts_not_school.select('count').to_numpy().flatten())
prop_school = (counts_school.select('count_sequences').to_numpy().flatten() / 
                   cases_school.select('count').to_numpy().flatten())
dates = counts_school.select('PRDATE_ADJUSTED').to_numpy().flatten().astype(str)
dates_plot = [datetime.strptime(d.split('T')[0], '%Y-%m-%d') for d in dates.flatten()]

In [ ]:
import pygam
from pygam import s, f


X = np.arange(len(dates_plot))
prop_not_school_gam = np.array(prop_not_school)
gam = pygam.LinearGAM(s(0)).fit(X, prop_not_school_gam)
    

for i, term in enumerate(gam.terms):

    if term.isintercept:
        continue
    XX = gam.generate_X_grid(term=i)
    pdep_not_school, confi_not_school = gam.partial_dependence(term=i, X=XX, width=0.95)
    intercept_not_school = gam.coef_[-1]
    
    
prop_school_gam = np.array(prop_school)
gam = pygam.LinearGAM(s(0)).fit(X, prop_school_gam)
for i, term in enumerate(gam.terms):

    if term.isintercept:
        continue
    XX = gam.generate_X_grid(term=i)
    pdep_school, confi_school = gam.partial_dependence(term=i, X=XX, width=0.95)
    intercept_school = gam.coef_[-1]
    

In [ ]:
confis_school = (confi_school + intercept_school).T
lci_school = confis_school[0, :]
uci_school = confis_school[1, :]
confis_not_school = (confi_not_school + intercept_not_school).T
lci_not_school = confis_not_school[0, :]
uci_not_school = confis_not_school[1, :]

In [ ]:
cmp_vir = ['#440154FF', '#3B528BFF', '#21908CFF', '#5DC863FF', '#FDE725FF']
date_labs = [d.date().strftime('%b\n%Y') for d in pd.date_range(start='9/1/2020', end ='1/1/2022', freq='D')]

fig = plt.figure()
fig.set_figwidth(15)
ax = fig.gca()

plt.plot(XX, pdep_not_school + intercept_not_school, color = cmp_vir[-3], label = 'Not in school', lw = 2)
plt.fill_between(XX.flatten(), lci_not_school, uci_not_school, color = cmp_vir[-3], alpha = 0.2)
plt.plot(XX, pdep_school + intercept_school, color = cmp_vir[-2], label = 'In school', lw = 2)
plt.fill_between(XX.flatten(), lci_school, uci_school, color = cmp_vir[-2], alpha = 0.2)
ax.xaxis.set_ticks(np.arange(len(date_labs)).astype(int), date_labs)
    
ax.xaxis.set_major_locator(MonthLocator(np.arange(12).astype(int) + 1))
plt.ylabel('Proportion of Cases')
plt.xlim([0, 365])
plt.ylim([0, 1])
plt.xlabel('Date')
plt.title('Daily Proportion of positive PCRs that are sequenced - Schools')
plt.grid(alpha = 0.5)
plt.legend(title = 'Setting')
# plt.savefig('../Figures/sequence_prop_schools.png')
# plt.plot(dates_plot, prop_school)
# plt.plot(dates_plot, prop_not_school)

In [ ]:
pcrs_work_ids = pcrs_df.join(df_work, on = 'PERSON_ID', how = 'left')
pcrs_not_work = pcrs_work_ids.filter(pl.col('ARB_NR').is_null())
pcrs_work = pcrs_work_ids.filter(~pl.col('ARB_NR').is_null())
people_tests_work = pcrs_work.group_by('PRDATE_ADJUSTED').len().sort('PRDATE_ADJUSTED').rename({'len': 'count'})[2:]
people_tests_not_work = pcrs_not_work.group_by('PRDATE_ADJUSTED').len().sort('PRDATE_ADJUSTED').rename({'len' : 'count'})[1:-5]

In [ ]:
work_metadata = metadata_all_ages.join(df_work, on ='PERSON_ID', how = 'left').with_columns(pl.col('DATESAMPLING').
                                                                                            str.strptime(pl.Date))

sequences_work = work_metadata.filter(~pl.col('ARB_NR').is_null())
sequences_not_work = work_metadata.filter(pl.col('ARB_NR').is_null())
people_sequences_work = sequences_work.group_by('DATESAMPLING').len().sort('DATESAMPLING').rename({'len' : 'count'})
people_sequences_not_work = sequences_not_work.group_by('DATESAMPLING').len().sort('DATESAMPLING').rename({'len' : 'count'})


counts_not_work = people_tests_not_work.with_columns(pl.col('PRDATE_ADJUSTED').cast(pl.Date)).join(people_sequences_not_work,
                                                                            left_on = 'PRDATE_ADJUSTED', 
                                                                           right_on = 'DATESAMPLING', 
                                                                            how = 'left', 
                                                                           suffix = '_sequences').fill_null(0)

counts_work = people_tests_work.with_columns(pl.col('PRDATE_ADJUSTED').cast(pl.Date)).join(people_sequences_work,
                                                                            left_on = 'PRDATE_ADJUSTED', 
                                                                           right_on = 'DATESAMPLING', 
                                                                            how = 'left', 
                                                                           suffix = '_sequences').fill_null(0)

In [ ]:
prop_not_work = (counts_not_work.select('count_sequences').to_numpy().flatten() / 
                counts_not_work.select('count').to_numpy().flatten())
prop_work = (counts_work.select('count_sequences').to_numpy().flatten() / 
                counts_work.select('count').to_numpy().flatten())

In [ ]:
X = np.arange(len(dates_plot))
prop_not_work_gam = np.array(prop_not_work)
gam = pygam.LinearGAM(s(0)).fit(X[1:], prop_not_work_gam)
    

for i, term in enumerate(gam.terms):

    if term.isintercept:
        continue
    XX = gam.generate_X_grid(term=i)
    pdep_not_work, confi_not_work = gam.partial_dependence(term=i, X=XX, width=0.95)
    intercept_not_work = gam.coef_[-1]
    
    
prop_work_gam = np.array(prop_work)
gam = pygam.LinearGAM(s(0)).fit(X, prop_work_gam)
for i, term in enumerate(gam.terms):

    if term.isintercept:
        continue
    XX = gam.generate_X_grid(term=i)
    pdep_work, confi_work = gam.partial_dependence(term=i, X=XX, width=0.95)
    intercept_work = gam.coef_[-1]

    
    
confis_work = (confi_work + intercept_work).T
lci_work = confis_work[0, :]
uci_work = confis_work[1, :]
confis_not_work = (confi_not_work + intercept_not_work).T
lci_not_work = confis_not_work[0, :]
uci_not_work = confis_not_work[1, :]

In [ ]:
fig = plt.figure()
fig.set_figwidth(15)
ax = fig.gca()

plt.plot(XX, pdep_not_work + intercept_not_work, color = cmp_vir[-3], label = 'Not in Work', lw = 2)
plt.fill_between(XX.flatten(), lci_not_work, uci_not_work, color = cmp_vir[-3], alpha = 0.2)
plt.plot(XX, pdep_work + intercept_work, color = cmp_vir[-5], label = 'In Work', lw = 2)
plt.fill_between(XX.flatten(), lci_work, uci_work, color = cmp_vir[-5], alpha = 0.2)
ax.xaxis.set_ticks(np.arange(488).astype(int), date_labs)
    
ax.xaxis.set_major_locator(MonthLocator(np.arange(12).astype(int) + 1))
plt.ylabel('Proportion of Cases')
# plt.xlim([0, 365])
plt.ylim([0, 1])
plt.xlabel('Date')
plt.title('Daily Proportion of positive PCRs that are sequenced - Workplaces')
plt.grid(alpha = 0.5)
plt.legend(title = 'Setting:')


In [ ]:
run = 1

if run:
    def query_tests(age_limit_low=0, age_limit_high=15): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY TESTS (POSITIVE OR NEGATIVE) WITHIN AN AGE LIMIT HERE
        '''
        return([q1])

    def retrieve_data(query):
        # extract data from DB for all pids
        oracledb.init_oracle_client()
        with oracledb.connect(user='', dsn='') as con:
            with con.cursor() as cur:
                out = [np.asarray(cur.execute(qq).fetchall()) for qq in query()]
                cols = cur.description
        return(out, cols)



    data_tests, columns_tests = retrieve_data(query_tests)
    cols_tests = ['PRDATE_ADJUSTED', 'Tests']

    df_tests = pd.DataFrame(data_tests[0], columns=cols_tests)
    df_tests = pl.from_pandas(df_tests)

In [ ]:
age_groups = [(0, 10), (11, 18), (19, 39), (40, 59), (60, 150)]
age_groups_str = ['0-10', '11-18', '19-39', '40-59', '60+']
a = 0
test_df_dict = {}
for ages in tqdm(age_groups):
    a_low, a_high = ages
    
    def query_tests(age_limit_low=a_low, age_limit_high=a_high): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY TESTS (POSITIVE OR NEGATIVE) WITHIN AN AGE LIMIT HERE
        '''
        return([q1])

    data_tests, columns_tests = retrieve_data(query_tests)
    cols_tests = ['PRDATE_ADJUSTED', 'Tests']
    
    # Convert query output to polars df via pandas
    df_tests = pd.DataFrame(data_tests[0], columns=cols_tests)
    df_tests = pl.from_pandas(df_tests)
    
    test_df_dict[age_groups_str[a]] = df_tests
    a += 1
    

In [ ]:
df_lifelines = df_lifelines.with_columns(Age_in_2021 = 
                                         ((pl.date(2021, 12, 31) - pl.col('BIRTHDAY')).dt.total_days() / 365).cast(pl.Int64))

df_lifelines = df_lifelines.with_columns(Age_group = 
    pl.when(pl.col('Age_in_2021') >=60).then(
        pl.lit('60+')
    ).otherwise(pl.when(pl.col('Age_in_2021').is_between(40, 59)).then(
    pl.lit('40-59')
    ).otherwise(pl.when(pl.col('Age_in_2021').is_between(19, 39)).then(
    pl.lit('19-39')
    ).otherwise(pl.when(pl.col('Age_in_2021').is_between(11, 18)).then(
        pl.lit('11-18'))
                 .otherwise(pl.when(pl.col('Age_in_2021').is_between(0, 10)).then(
                 pl.lit('0-10'))
               )))))

In [ ]:
pop_sizes = dict(df_lifelines.group_by('Age_group').len().rename({'len' : 'count'}).to_numpy())

In [ ]:
def get_gam(X, y):
    gam = pygam.LinearGAM(s(0)).fit(X, y)
    

    for i, term in enumerate(gam.terms):

        if term.isintercept:
            continue
        XX = gam.generate_X_grid(term=i)
        pdep_gam, confi_gam = gam.partial_dependence(term=i, X=XX, width=0.95)
        intercept_gam = gam.coef_[-1]
    return {'x_axis' : XX, 
            'fit' : pdep_gam+intercept_gam, 
            'CI' : confi_gam+intercept_gam, 
            'intercept' : intercept_gam}


    

In [ ]:
import scienceplots
plt.style.use('science')
date_labs = [d.date().strftime('%b\n%Y') for d in pd.date_range(start='9/1/2020', end ='1/1/2022', freq='D')]

fig = plt.figure(figsize = (15, 5))
ax = fig.gca()
for age_group_str in age_groups_str:
    pop_size = pop_sizes[age_group_str]
    test_df = test_df_dict[age_group_str]
    plot_dates = test_df.with_columns(pl.col('PRDATE_ADJUSTED')
                                      .cast(pl.Date)).select('PRDATE_ADJUSTED').to_numpy().flatten()
    tests_by_day = test_df.select('Tests').to_numpy().flatten()
    gam_dict = get_gam(np.arange(len(plot_dates)), tests_by_day / pop_size)
    
    x_axis = gam_dict['x_axis']
    fit = gam_dict['fit']
    confi_l = gam_dict['CI'][:, 0].flatten()
    confi_u = gam_dict['CI'][:, 1].flatten()
    
    plt.plot(x_axis, fit, label  = age_group_str)
    plt.fill_between(x_axis.flatten(), confi_l, confi_u, alpha = 0.2)
ax.xaxis.set_ticks(np.arange(len(plot_dates)).astype(int), plot_dates)
ax.xaxis.set_major_locator(MonthLocator(bymonthday = 1))
plt.legend()

In [ ]:
a = 0
pos_test_df_dict = {}
for ages in tqdm(age_groups):
    a_low, a_high = ages
    
    def query_pos_tests(age_limit_low=a_low, age_limit_high=a_high): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY TESTS (POSITIVE ONLY) WITHIN AN AGE LIMIT HERE
        '''
        return([q1])

    data_pos_tests, columns_pos_tests = retrieve_data(query_pos_tests)
    cols_pos_tests = ['PRDATE_ADJUSTED', 'Tests']
    
    # Convert query output to polars df via pandas
    df_pos_tests = pd.DataFrame(data_pos_tests[0], columns=cols_pos_tests)
    df_pos_tests = pl.from_pandas(df_pos_tests)
    
    pos_test_df_dict[age_groups_str[a]] = df_pos_tests
    a += 1
    

In [ ]:
fig = plt.figure(figsize = (15, 5))
ax = fig.gca()
for age_group_str in age_groups_str:
    pop_size = pop_sizes[age_group_str]
    test_df = test_df_dict[age_group_str]
    pos_test_df = pos_test_df_dict[age_group_str]
    plot_dates = test_df.with_columns(pl.col('PRDATE_ADJUSTED').cast(pl.Date)).select('PRDATE_ADJUSTED').to_numpy().flatten()
    tests_by_day = test_df.select('Tests').to_numpy().flatten()
    pos_tests_by_day = pos_test_df.select('Tests').to_numpy().flatten()
    gam_dict = get_gam(np.arange(len(plot_dates)), pos_tests_by_day / tests_by_day)
    
    x_axis = gam_dict['x_axis']
    fit = gam_dict['fit']
    confi_l = gam_dict['CI'][:, 0].flatten()
    confi_u = gam_dict['CI'][:, 1].flatten()
    
    plt.plot(x_axis, fit, label  = age_group_str)
    plt.fill_between(x_axis.flatten(), confi_l, confi_u, alpha = 0.2)
ax.xaxis.set_ticks(np.arange(len(date_labs)).astype(int), date_labs)
ax.xaxis.set_major_locator(MonthLocator(interval = 1))
# plt.xlim([np.datetime64('2020-09-01'), np.datetime64('2022-01-01')])
plt.legend(title = 'Age group:')

In [ ]:
def query_all_tests(): 
        q1 = f'''
        QUERY ALL TESTS HERE
        '''
        return([q1])
    
def query_all_pos_tests(): 
        q1 = f'''
        QUERY ALL POSITIVE TESTS HERE
        '''
        return([q1])
    
data_all_tests, columns_all_tests = retrieve_data(query_all_tests)
cols_all_tests = ['PRDATE_ADJUSTED', 'Tests']
df_all_tests = pd.DataFrame(data_all_tests[0], columns=cols_all_tests)
df_all_tests = pl.from_pandas(df_all_tests)
print('Finished reading all tests')

data_all_pos_tests, columns_all_pos_tests = retrieve_data(query_all_pos_tests)
cols_all_pos_tests = ['PRDATE_ADJUSTED', 'Tests']
df_all_pos_tests = pd.DataFrame(data_all_pos_tests[0], columns=cols_all_pos_tests)
df_all_pos_tests = pl.from_pandas(df_all_pos_tests)
print('Finished reading all positive tests')


In [ ]:
def query_school_tests(): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY ALL TESTS WITHIN SCHOOLS HERE
        '''
        return([q1])
    
 
    
data_school_tests, columns_school_tests = retrieve_data(query_school_tests)
cols_school_tests = ['PRDATE_ADJUSTED', 'Tests']
df_school_tests = pd.DataFrame(data_school_tests[0], columns=cols_school_tests)
df_school_tests = pl.from_pandas(df_school_tests)
print('Finished reading school tests')


def query_pos_school_tests(): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY POSITIVE SCHOOL TESTS HERE
        '''
        return([q1])
    
    
data_pos_school_tests, columns_pos_school_tests = retrieve_data(query_pos_school_tests)
cols_pos_school_tests = ['PRDATE_ADJUSTED', 'Tests']
df_pos_school_tests = pd.DataFrame(data_pos_school_tests[0], columns=cols_pos_school_tests)
df_pos_school_tests = pl.from_pandas(df_pos_school_tests)
print('Finished reading school positive tests')


In [ ]:
def query_work_tests(): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY WORKPLACE TESTS HERE
        '''
        return([q1])
    

    
data_work_tests, columns_work_tests = retrieve_data(query_work_tests)
cols_work_tests = ['PRDATE_ADJUSTED', 'Tests']
df_work_tests = pd.DataFrame(data_work_tests[0], columns=cols_work_tests)
df_work_tests = pl.from_pandas(df_work_tests)
print('Finished reading work tests')




def query_pos_work_tests(): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY POSITIVE WORKPLACE TESTS HERE 
        '''
        return([q1])
 
    
    
data_pos_work_tests, columns_pos_work_tests = retrieve_data(query_pos_work_tests)
cols_pos_work_tests = ['PRDATE_ADJUSTED', 'Tests']
df_pos_work_tests = pd.DataFrame(data_pos_work_tests[0], columns=cols_pos_work_tests)
df_pos_work_tests = pl.from_pandas(df_pos_work_tests)
print('Finished reading work positive tests')


In [ ]:
school_pop = 1368329 # Number of people in school during the time period
not_school_pop = sum(list(pop_sizes.values())) - school_pop
all_tests = df_all_tests.select('Tests').to_numpy().flatten()
school_tests = df_school_tests.select('Tests').to_numpy().flatten()
not_school_tests = all_tests - school_tests

all_pos_tests = df_all_pos_tests.select('Tests').to_numpy().flatten()
school_pos_tests = df_pos_school_tests.select('Tests').to_numpy().flatten()
not_school_pos_tests = all_pos_tests - school_pos_tests

school_test_prop = school_tests / school_pop

not_school_test_prop = (not_school_tests / not_school_pop)

plt.plot(school_test_prop , label = 'School' )
plt.plot(not_school_test_prop, label = 'Not School')
plt.legend()

In [ ]:
school_pos_prop = (school_pos_tests / school_tests)
not_school_pos_prop = (not_school_pos_tests / not_school_tests )
plt.plot(school_pos_prop, label = 'School' )
plt.plot(not_school_pos_prop, label = 'Not School')
plt.legend()

In [ ]:
work_pop = 3222671  # Number of people in work during the time period

not_work_pop = sum(list(pop_sizes.values())) - work_pop
work_tests = df_work_tests.select('Tests').to_numpy().flatten()
not_work_tests = all_tests - work_tests

work_pos_tests = df_pos_work_tests.select('Tests').to_numpy().flatten()
not_work_pos_tests = all_pos_tests - work_pos_tests

work_test_prop = work_tests / work_pop

not_work_test_prop = (not_work_tests / not_work_pop)

plt.plot(work_test_prop, label = 'Work'  )
plt.plot(not_work_test_prop, label = 'Not work')
plt.legend()

In [ ]:
work_pos_prop = (work_pos_tests / work_tests)
not_work_pos_prop = (not_work_pos_tests / not_work_tests )
plt.plot(work_pos_prop, label = 'work' )
plt.plot(not_work_pos_prop, label = 'Not work')
plt.legend()

In [ ]:
df_episodes = pl.from_pandas(df_full)
df_episodes_first_group = df_episodes.sort('PRDATE_ADJUSTED').group_by('PERSON_ID').first()
rowid_list = list(df_episodes_first_group.select('ROWID').to_numpy().flatten())
df_episodes_first = df_episodes.join(df_episodes_first_group
                 .select(pl.col('PERSON_ID', 'PRDATE_ADJUSTED')), on = 'PERSON_ID', how = 'left', suffix = '_first')
df_episodes_first = df_episodes_first.with_columns(Time_since_first =(( pl.col('PRDATE_ADJUSTED') -
                                                  pl.col('PRDATE_ADJUSTED_first')).dt.total_days() / 60).cast(pl.Int64))
df_episodes_first = df_episodes_first.filter(~pl.col('ROWID').is_in(rowid_list))
for i in tqdm(range(8)):
    df_episodes_first_group =df_episodes_first.sort('PRDATE_ADJUSTED').group_by('PERSON_ID').first()
    df_episodes_first = df_episodes_first.drop('PRDATE_ADJUSTED_first').join(df_episodes_first_group
                 .select(pl.col('PERSON_ID', 'PRDATE_ADJUSTED')), on = 'PERSON_ID', how = 'left', suffix = '_first')
    df_episodes_first = df_episodes_first.with_columns(Time_since_first =(( pl.col('PRDATE_ADJUSTED') -
                                                      pl.col('PRDATE_ADJUSTED_first')).dt.total_days() / 60).cast(pl.Int64))
    rowid_list += list(df_episodes_first_group.select('ROWID').to_numpy().flatten())
    df_episodes_first = df_episodes_first.filter(~pl.col('ROWID').is_in(rowid_list))

# df_episodes_first

In [ ]:
len(df_episodes) - len(df_episodes.filter(pl.col('ROWID').is_in(rowid_list)))
df_episodes_unique = df_episodes.filter(pl.col('ROWID').is_in(rowid_list))

In [ ]:
start_date = pl.date(2020, 9, 1)
end_date =  pl.date(2022, 1, 1)
df_episodes_unique = df_episodes_unique.with_columns(Age_group = 
    pl.when(pl.col('AGE_EPIMIBA') >=60).then(
        pl.lit('60+')
    ).otherwise(pl.when(pl.col('AGE_EPIMIBA').is_between(40, 59)).then(
    pl.lit('40-59')
    ).otherwise(pl.when(pl.col('AGE_EPIMIBA').is_between(19, 39)).then(
    pl.lit('19-39')
    ).otherwise(pl.when(pl.col('AGE_EPIMIBA').is_between(11, 18)).then(
        pl.lit('11-18'))
                 .otherwise(pl.when(pl.col('AGE_EPIMIBA').is_between(0, 10)).then(
                 pl.lit('0-10'))
               )))))
df_num_cases_age_dict = {}
for age_group_str in age_groups_str:
    df_episodes_unique_age = df_episodes_unique.filter(pl.col('Age_group')== age_group_str)
    df_num_cases_age = df_episodes_unique_age.group_by(pl.col('PRDATE_ADJUSTED')).len().sort('PRDATE_ADJUSTED').rename({'len' : 'count'})
    df_num_cases_age = df_num_cases_age.filter(pl.col('PRDATE_ADJUSTED').is_between(start_date, end_date))
    df_num_cases_age_dict[age_group_str] = df_num_cases_age


In [ ]:
df_pos_pcrs

In [ ]:
age_groups = [(0, 10), (11, 18), (19, 39), (40, 59), (60, 150)]
age_groups_str = ['0-10', '11-18', '19-39', '40-59', '60+']
a = 0
pos_pcr_df_dict = {}
for ages in tqdm(age_groups):
    a_low, a_high = ages
    
    def query_pos_pcrs(age_limit_low=a_low, age_limit_high=a_high): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        QUERY POSITIVE PCRS HERE
        '''
        return([q1])

    data_pos_pcrs, columns_pos_pcrs = retrieve_data(query_pos_pcrs)
    cols_pos_pcrs = ['PRDATE_ADJUSTED', 'Tests']
    
    # Convert query output to polars df via pandas
    df_pos_pcrs = pd.DataFrame(data_pos_pcrs[0], columns=cols_pos_pcrs)
    df_pos_pcrs = pl.from_pandas(df_pos_pcrs)
    
    pos_pcr_df_dict[age_groups_str[a]] = df_pos_pcrs
    a += 1
    

In [ ]:
dates_df = metadata_all_ages.select('DATESAMPLING').unique().sort('DATESAMPLING')
for age_group_str in age_groups_str:
    metadata_all_ages_age_group = metadata_all_ages.filter(pl.col('Age_group')==age_group_str)
    pos_pcrs_age_group = pos_pcr_df_dict[age_group_str].select('Tests').to_numpy().flatten()
    seqs_age_group = metadata_all_ages_age_group.group_by('DATESAMPLING').len().rename({'len' : 'count'})
    seqs_age_group = dates_df.join(seqs_age_group, on = 'DATESAMPLING', how = 'left').fill_null(0)
    gam_dict = get_gam(np.arange(len(plot_dates)), (seqs_age_group.select('count').to_numpy().flatten()
                                                    / pos_pcrs_age_group))

    x_axis = gam_dict['x_axis']
    fit = gam_dict['fit']
    confi_l = gam_dict['CI'][:, 0].flatten()
    confi_u = gam_dict['CI'][:, 1].flatten()
    
    plt.plot(x_axis, fit, label  = age_group_str)
    plt.fill_between(x_axis.flatten(), confi_l, confi_u, alpha = 0.2)
plt.legend()

In [ ]:
age_ranges = [[0, 10], [11, 18], [19, 39], [40, 59], [60, 150]]
age_groups_str
metadata_ages = metadata.join(df_lifelines, on = 'PERSON_ID', how = 'left')
metadata_ages = metadata_ages.with_columns(Age_at_testing = 
                                           ((pl.col('SampleDateTime')
                                            .str.strptime(pl.Datetime, '%Y-%m-%d %H.%M.%S%.f').cast(pl.Date)-
                                           pl.col('BIRTHDAY').cast(pl.Date)).dt.total_days()/365).cast(pl.Int64))
date_df = df_episodes_unique.select('PRDATE_ADJUSTED').unique().sort('PRDATE_ADJUSTED')
cases_age = {}
cases_by_age = np.zeros((492, len(age_groups_str)))
for i, ran in enumerate(age_ranges):
    lb, ub = ran[0], ran[1]
    cases_age_df = df_episodes_unique.filter(pl.col('AGE_EPIMIBA').is_between(lb, ub))
    cases_age_count = cases_age_df.group_by('PRDATE_ADJUSTED').len().sort('PRDATE_ADJUSTED').rename({'len' : 'count'})
    cases_age_count = date_df.join(cases_age_count, on = 'PRDATE_ADJUSTED', how = 'left').fill_null(0)
    cases_age[age_groups_str[i]] = cases_age_count
    cases_age_vec = cases_age_count.select('count').to_numpy().flatten()
    cases_by_age[:, i] = cases_age_vec
    
    seqs_age_df = df_episodes_unique.filter(pl.col('AGE_EPIMIBA').is_between(lb, ub))
    seqs_age_count = seqs_age_df.group_by('PRDATE_ADJUSTED').len().sort('PRDATE_ADJUSTED').rename({'len' : 'count'})
    seqs_age_count = date_df.join(cases_age_count, on = 'PRDATE_ADJUSTED', how = 'left').fill_null(0)


In [ ]:
fig, axs = plt.subplots(3, 3, figsize = (15, 15), sharex = True)

#1st row: age groups
for age_group_str in age_groups_str:
    # Testing proportion
    pop_size = pop_sizes[age_group_str]
    test_df = test_df_dict[age_group_str]
    num_cases_df = df_num_cases_age_dict[age_group_str]
    plot_dates = test_df.with_columns(pl.col('PRDATE_ADJUSTED')
                                      .cast(pl.Date)).select('PRDATE_ADJUSTED').to_numpy().flatten()
    tests_by_day = test_df.select('Tests').to_numpy().flatten()
    gam_dict = get_gam(np.arange(len(plot_dates)), tests_by_day / pop_size)
    
    x_axis = gam_dict['x_axis']
    fit = gam_dict['fit']
    confi_l = gam_dict['CI'][:, 0].flatten()
    confi_u = gam_dict['CI'][:, 1].flatten()
    
    axs[0,0].plot(x_axis, fit, label  = age_group_str, linewidth = 2)
    axs[0,0].fill_between(x_axis.flatten(), confi_l, confi_u, alpha = 0.2)
    
    # Positive testing proportion
    pop_size = pop_sizes[age_group_str]
    test_df = test_df_dict[age_group_str]
    pos_test_df = pos_test_df_dict[age_group_str]
    plot_dates = test_df.with_columns(pl.col('PRDATE_ADJUSTED')
                                      .cast(pl.Date)).select('PRDATE_ADJUSTED').to_numpy().flatten()
    tests_by_day = test_df.select('Tests').to_numpy().flatten()
    pos_tests_by_day = pos_test_df.select('Tests').to_numpy().flatten()
    gam_dict = get_gam(np.arange(len(plot_dates)), pos_tests_by_day / tests_by_day)
    
    x_axis = gam_dict['x_axis']
    fit = gam_dict['fit']
    confi_l = gam_dict['CI'][:, 0].flatten()
    confi_u = gam_dict['CI'][:, 1].flatten()
    
    axs[0, 1].plot(x_axis, fit, label  = age_group_str, linewidth = 2)
    axs[0, 1].fill_between(x_axis.flatten(), confi_l, confi_u, alpha = 0.2)
    
    
    metadata_all_ages_age_group = metadata_all_ages.filter(pl.col('Age_group')==age_group_str)
    pos_pcrs_age_group = pos_pcr_df_dict[age_group_str].select('Tests').to_numpy().flatten()
    seqs_age_group = metadata_all_ages_age_group.group_by('DATESAMPLING').len().rename({'len' : 'count'})
    seqs_age_group = dates_df.join(seqs_age_group, on = 'DATESAMPLING', how = 'left').fill_null(0)

    gam_dict = get_gam(np.arange(len(plot_dates)), (seqs_age_group.select('count').to_numpy().flatten()
                                                    / pos_pcrs_age_group))

    x_axis = gam_dict['x_axis']
    fit = gam_dict['fit']
    confi_l = gam_dict['CI'][:, 0].flatten()
    confi_u = gam_dict['CI'][:, 1].flatten()
    
    axs[0, 2].plot(x_axis, fit, label  = age_group_str)
    axs[0, 2].fill_between(x_axis.flatten(), confi_l, confi_u, alpha = 0.2)
axs[0, 2].legend()

#2nd row: schools

school_pos_tests_gam = get_gam(np.arange(len(plot_dates)), school_pos_tests / school_tests)
school_tests_gam = get_gam(np.arange(len(plot_dates)), school_tests / school_pop)
not_school_pos_tests_gam = get_gam(np.arange(len(plot_dates)), not_school_pos_tests / not_school_tests)
not_school_tests_gam = get_gam(np.arange(len(plot_dates)), not_school_tests / not_school_pop)

XX = school_pos_tests_gam['x_axis'].flatten()
school_pos_tests_gam_fit = school_pos_tests_gam['fit']
school_tests_gam_fit = school_tests_gam['fit']
not_school_pos_tests_gam_fit = not_school_pos_tests_gam['fit']
not_school_tests_gam_fit = not_school_tests_gam['fit']

confi_l_spt = school_pos_tests_gam['CI'][:, 0].flatten()
confi_u_spt = school_pos_tests_gam['CI'][:, 1].flatten()
confi_l_st = school_tests_gam['CI'][:, 0].flatten()
confi_u_st = school_tests_gam['CI'][:, 1].flatten()
confi_l_nspt = not_school_pos_tests_gam['CI'][:, 0].flatten()
confi_u_nspt = not_school_pos_tests_gam['CI'][:, 1].flatten()
confi_l_nst = not_school_tests_gam['CI'][:, 0].flatten()
confi_u_nst = not_school_tests_gam['CI'][:, 1].flatten()

axs[1, 0].plot(XX, school_tests_gam_fit, label = 'School', color = cmp_vir[-5], linewidth = 2)
axs[1, 0].plot(XX, not_school_tests_gam_fit, label = 'Not in School', color = cmp_vir[-2], linewidth = 2)
axs[1, 0].fill_between(XX, confi_l_st, confi_u_st, alpha = 0.2, color =  cmp_vir[-5])
axs[1, 0].fill_between(XX, confi_l_nst, confi_u_nst, alpha = 0.2, color = cmp_vir[-2])

axs[1, 1].plot(XX, school_pos_tests_gam_fit, label = 'School', color = cmp_vir[-5], linewidth = 2)
axs[1, 1].plot(XX, not_school_pos_tests_gam_fit, label = 'Not in School', color = cmp_vir[-2], linewidth = 2)
axs[1, 1].fill_between(XX, confi_l_spt, confi_u_spt, alpha = 0.2, color =  cmp_vir[-5])
axs[1, 1].fill_between(XX, confi_l_nspt, confi_u_nspt, alpha = 0.2, color = cmp_vir[-2])

axs[1, 2].plot(XX, pdep_not_school + intercept_not_school, color = cmp_vir[-2], label = 'Not in school', lw = 2)
axs[1, 2].fill_between(XX, lci_not_school, uci_not_school, color = cmp_vir[-2], alpha = 0.2)
axs[1, 2].plot(XX, pdep_school + intercept_school, color = cmp_vir[-5], label = 'In school', lw = 2)
axs[1, 2].fill_between(XX, lci_school, uci_school, color = cmp_vir[-5], alpha = 0.2)

axs[1, 2].legend()


#3rd row: workplaces

work_pos_tests_gam = get_gam(np.arange(len(plot_dates)), work_pos_tests / work_tests)
work_tests_gam = get_gam(np.arange(len(plot_dates)), work_tests / work_pop)
not_work_pos_tests_gam = get_gam(np.arange(len(plot_dates)), not_work_pos_tests / not_work_tests)
not_work_tests_gam = get_gam(np.arange(len(plot_dates)), not_work_tests / not_work_pop)

XX = work_pos_tests_gam['x_axis'].flatten()
work_pos_tests_gam_fit = work_pos_tests_gam['fit']
work_tests_gam_fit = work_tests_gam['fit']
not_work_pos_tests_gam_fit = not_work_pos_tests_gam['fit']
not_work_tests_gam_fit = not_work_tests_gam['fit']

confi_l_wpt = work_pos_tests_gam['CI'][:, 0].flatten()
confi_u_wpt = work_pos_tests_gam['CI'][:, 1].flatten()
confi_l_wt = work_tests_gam['CI'][:, 0].flatten()
confi_u_wt = work_tests_gam['CI'][:, 1].flatten()
confi_l_nwpt = not_work_pos_tests_gam['CI'][:, 0].flatten()
confi_u_nwpt = not_work_pos_tests_gam['CI'][:, 1].flatten()
confi_l_nwt = not_work_tests_gam['CI'][:, 0].flatten()
confi_u_nwt = not_work_tests_gam['CI'][:, 1].flatten()

axs[2, 0].plot(XX, work_tests_gam_fit, label = 'Work', color = cmp_vir[-5], linewidth = 2)
axs[2, 0].plot(XX, not_work_tests_gam_fit, label = 'Not in Work', color = cmp_vir[-2], linewidth = 2)
axs[2, 0].fill_between(XX, confi_l_wt, confi_u_wt, alpha = 0.2, color =  cmp_vir[-5])
axs[2, 0].fill_between(XX, confi_l_nwt, confi_u_nwt, alpha = 0.2, color = cmp_vir[-2])

axs[2, 1].plot(XX, work_pos_tests_gam_fit, label = 'Work', color = cmp_vir[-5], linewidth = 2)
axs[2, 1].plot(XX, not_work_pos_tests_gam_fit, label = 'Not in Work', color = cmp_vir[-2], linewidth = 2)
axs[2, 1].fill_between(XX, confi_l_wpt, confi_u_wpt, alpha = 0.2, color =  cmp_vir[-5])
axs[2, 1].fill_between(XX, confi_l_nwpt, confi_u_nwpt, alpha = 0.2, color = cmp_vir[-2])

axs[2, 2].plot(XX, pdep_not_work + intercept_not_work, color = cmp_vir[-2], label = 'Not in Work', lw = 2)
axs[2, 2].fill_between(XX, lci_not_work, uci_not_work, color = cmp_vir[-2], alpha = 0.2)
axs[2, 2].plot(XX, pdep_work + intercept_work, color = cmp_vir[-5], label = 'In Work', lw = 2)
axs[2, 2].fill_between(XX, lci_work, uci_work, color = cmp_vir[-5], alpha = 0.2)

axs[2, 2].legend()


for i in range(3):
    axs[i, 0].set_ylim([0, 0.2])
    axs[i, 1].set_ylim([0, 0.1])
    axs[i, 2].set_ylim([0, 1])
    
    axs[-1, i].xaxis.set_ticks(np.arange(488).astype(int), date_labs)
    
    axs[-1, i].xaxis.set_major_locator(MonthLocator(interval=3))
    
#     axs[-1, 0]
    for j in range(3):
        axs[i, j].grid(alpha = 0.5)
#         axs[i, j].label_outer()
fig.supylabel('Proportion \n', fontsize = 16)
fig.supxlabel('Date', fontsize = 16)
fig.suptitle('Daily Sampling Proportions Among Different Groups \n ', fontsize = 16)
axs[0, 0].set_title('Proportion Taking a Test', fontsize = 12)
axs[0, 1].set_title('Proportion of Tests that are Positive', fontsize = 12)
axs[0, 2].set_title('Proportion of Positive PCRs that are Sequenced', fontsize = 12)

axs[0, 0].set_ylabel('By Age Group', fontsize = 12)
axs[1, 0].set_ylabel('Among School Groups', fontsize = 12)
axs[2, 0].set_ylabel('Among People in Work', fontsize = 12)


fig.tight_layout()
# plt.savefig('../Figures/sampling_proportions_group_figure.png')

In [ ]:
num_seqs_in_study = 0
num_pos_pcrs_in_study = 0
for age_group_str in age_groups_str:
    metadata_all_ages_age_group = metadata_all_ages.filter(pl.col('Age_group')==age_group_str)
    pos_pcrs_age_group = pos_pcr_df_dict[age_group_str].filter(pl.col('PRDATE_ADJUSTED') >=
                                                               datetime.strptime('2021-01-01', 
                                                                                 '%Y-%m-%d')).select('Tests').to_numpy().flatten()
    seqs_age_group = metadata_all_ages_age_group.group_by('DATESAMPLING').len().rename({'len' : 'count'})
    seqs_age_group = dates_df.join(seqs_age_group, on = 'DATESAMPLING', how = 'left').fill_null(0).filter(pl.col('DATESAMPLING') >= '2021-01-01')
    
    num_seqs_in_study += np.sum(seqs_age_group.select('count').to_numpy().flatten())
    num_pos_pcrs_in_study += np.sum(pos_pcrs_age_group)
